# Séance S01 — 01a : découvrir les vraies données avec PySpark

Même logique que la simulation miniature (lire, regarder, filtrer, joindre, agréger, tracer), mais avec **Spark** et les **vraies** courses de taxis de New York : environ **3 millions** de lignes pour janvier 2025.

**Avant de commencer**
- Les données sont téléchargées : `bash scripts/telecharger_donnees.sh 2025-01`
- Docker tourne : `docker compose up -d`
- Ce notebook est branché sur le kernel Jupyter (Select Kernel → Existing Jupyter Server → `http://localhost:8888/?token=cours`)

**Comment travailler** : exécute les cellules **une par une** (Maj + Entrée), et remplace chaque `# TODO` par ton code, un exercice à la fois.

## Cellule 0 — Préparation (déjà faite)

On démarre une **session Spark** : c'est la « cuisine » où Spark va travailler. `local[*]` = utiliser tous les cœurs du processeur du conteneur.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("S01-decouvrir")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

# Le dépôt est visible dans le conteneur sous /home/jovyan/work
RAW = "/home/jovyan/work/data/raw"
print("Spark version :", spark.version)
print("Interface Spark : http://localhost:4040")

## Exercice 1 — Lire les courses

Charger `yellow_tripdata_2025-01.parquet` dans un DataFrame nommé `courses`.

*Indice* : `spark.read.parquet(f"{RAW}/nom_du_fichier.parquet")`  
*Équivalent pandas* : `pd.read_csv(...)`

In [ ]:
# TODO


## Exercice 2 — Regarder les données

a) Afficher le **schéma** (les colonnes et leurs types) — *pandas : `.dtypes`*  
b) Afficher **5 lignes** — *pandas : `.head()`*  
c) **Compter** les lignes — *pandas : `.shape[0]`*. Combien de courses en janvier 2025 ?

*Indices* : `.printSchema()`, `.show(5)`, `.count()`

In [ ]:
# TODO


## Exercice 3 — Véracité : repérer les lignes douteuses

Avec 3 millions de lignes, les erreurs ne sont plus 1 ou 2 : il y en a des milliers !

a) Combien de courses ont une `trip_distance` égale à 0 ?  
b) Combien ont un `total_amount` négatif ?  
c) Combien ont commencé (`tpep_pickup_datetime`) **avant** le 1er janvier 2025 ou **après** le 31 janvier ?  
d) Créer `courses_ok` qui ne garde que les courses valides, et compter ses lignes.

*Indices* : `courses.filter(F.col("trip_distance") == 0).count()`  
Pour combiner : `&` (et), `|` (ou), `~` (non), chaque condition **entre parenthèses**.

In [ ]:
# TODO


## Exercice 4 — Ajouter des colonnes calculées

Ajouter à `courses_ok` :
- `heure` : l'heure de départ — *indice : `F.hour("tpep_pickup_datetime")`*
- `pourboire_pct` : `tip_amount / fare_amount * 100`

*Indice* : `.withColumn("nom", calcul)` — *pandas : `df["nom"] = calcul`*

In [ ]:
# TODO


## Exercice 5 — Joindre les zones

a) Lire `taxi_zone_lookup.csv` dans `zones` — *indice : `spark.read.csv(..., header=True, inferSchema=True)`*  
b) Joindre `courses_ok` et `zones` pour obtenir le quartier (`Borough`) de la zone de **départ** (`PULocationID` ↔ `LocationID`).

*Indice* : `courses_ok.join(zones, courses_ok.PULocationID == zones.LocationID, "left")`  
*pandas : `.merge(..., left_on=..., right_on=..., how="left")`*

In [ ]:
# TODO


## Exercice 6 — Agréger par quartier

Pour chaque `Borough` de départ : nombre de courses, `total_amount` moyen, distance moyenne. Trier du plus grand nombre de courses au plus petit.

*Indices* : `.groupBy("Borough").agg(F.count("*").alias("nb_courses"), F.avg(...).alias(...), ...)`, puis `.orderBy(F.desc("nb_courses"))`

In [ ]:
# TODO


## Exercice 7 — Tracer un graphique

Le résultat de l'exercice 6 est **petit** (quelques lignes) : on peut le ramener dans **pandas** avec `.toPandas()`, puis tracer avec matplotlib.

Diagramme en barres du nombre de courses par quartier, avec titre et axes. L'enregistrer dans `/home/jovyan/work/mon_travail/S01/courses_par_quartier.png`.

⚠️ Ne jamais faire `.toPandas()` sur les 3 millions de lignes : seulement sur un résultat agrégé !

In [ ]:
# TODO


## Fin de séance

Arrêter la session Spark, puis, dans le terminal : `git add mon_travail`, `git commit`, `git push`, et `docker compose stop`.

In [ ]:
spark.stop()